### Messages

##### messages are fundamental unit of context for models in langchain. they represents the input and output of the models, carieng both the content and metadata needed to represent the state of the conversation when interacting with an llm. Messages are object that contain: 1. role: identifies the message type (eg. system,user) 2. content: represent the actual content of the messages (like text, images, audio documents,etc) 3. Metadata: Optional fields such as response information , message ids, and token usage, Langchain provides the standard message type that works across all model providers , ensuring consistent behaviour regardless of model bieng called.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ['GROQ_API_KEY']= os.getenv('GROQ_API_KEY')
model = init_chat_model("groq:qwen/qwen3-32b")


In [ ]:
model.invoke("what is cricket")# this is human message

AIMessage(content='<think>\nOkay, the user is asking "what is cricket". I need to provide a comprehensive answer. Let me start by outlining the key points.\n\nFirst, cricket is a sport played with a bat and ball. It\'s popular in many countries, especially in the Commonwealth. The objective is to score runs and outscore the opposing team. I should mention the basic rules: batting, bowling, wickets, and the different formats like Test, ODI, and T20.\n\nI should explain how the game is played, maybe break down the roles of the teams. Batsmen try to score runs by hitting the ball and running between wickets, while bowlers and fielders aim to take wickets. The wicket is three stumps with two bails on top. A batsman is out if the bails are dislodged, if they hit the ball with their bat and run, or other methods like run-outs.\n\nNext, the different formats: Test cricket is the longest, played over five days. ODIs are one-day matches with 50 overs each. T20 is the shortest, with 20 overs eac

### Text Prompt
#### Text Prompts are strings, idial for straingt forward generation tasks where you dont need to retain conversational history.

In [3]:
model.invoke("what is langchain")# simple text passed return we get ai message

AIMessage(content='<think>\nOkay, the user is asking "what is langchain." I need to provide a clear and concise explanation. Let me start by recalling what I know about LangChain.\n\nLangChain is a framework, right? It\'s used for developing applications with large language models (LLMs). Maybe I should mention that it helps in combining LLMs with other tools and data sources. The user might be a developer or someone interested in AI applications, so I should explain it in a way that highlights its utility and components.\n\nI should break down the key components. There are things like prompts, models (like GPT), memory, agents, and indexes. Each of these plays a role in building different parts of an application. For example, prompts are templates for the LLM to follow, models are the actual AI like GPT, memory allows the app to remember past interactions, agents can perform tasks using tools, and indexes help with searching through data.\n\nUse cases are also important. The user migh

#### Use Text Prompts when: You have a singel standalone request, You dont need conversational history, You want minimal code complexity. 

### Message Prompt: 
#### Alternatively you can pass in a list of messages to the model by providing a list of message object. Message types:
##### System Message: Tell the model how to behave and provide context for interactions
##### Human Message: User input and interactions with the model
##### AI Message: Responses generated by the model, including text content , tool calls and meta data
##### Tool message: represents the output of the calls

In [4]:
# simple list of messages
from langchain.messages import SystemMessage, HumanMessage
messages = [
    SystemMessage("You are a poetry expert"),
    HumanMessage("write a poem on cricket")
]
response = model.invoke(messages)
response.content

"<think>\nOkay, the user wants a poem about cricket. Let me start by thinking about the key elements of cricket. There's the field, the players, the bat and ball, the crowd. Maybe I can use some imagery related to the game. I should consider the structure—maybe a traditional rhyme scheme or something more free-form. Let me think of some cricketing terms: runs, wickets, bowlers, batters, spinners, fast bowlers. Also, emotions like tension during a close match, the thrill of a six, the disappointment of a missed catch. Perhaps include different aspects of the game—test matches, T20s, the history of cricket. Need to make it flow smoothly. Maybe start with the setting, then move to the players, the action, and the feelings it evokes. Let me try a quatrain with ABAB rhyme scheme. Check for rhythm, maybe iambic tetrameter. Avoid clichés, use fresh metaphors. Also, think about the audience—should the poem be for fans or more general? Probably safe to assume some familiarity with the game but 

In [7]:
# tool message
from langchain.messages import AIMessage
from langchain.messages import ToolMessage

# after a model makes a tool call, (Here we demonstrate mannually creating message for bravity)
ai_message =AIMessage(
    content=[],
    tool_calls = [{
        "name":"get_weather",
        "args":{"location":"Bangalore"},
        "id":"call_123"
    }]
)
# execute tool and create result message
weather_result = "Sunny, 33 degree F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id = "call_123"# must match call id
)
# continue conversation
messages = [
    HumanMessage("What is the weather in bangalore"),
    ai_message,# models tool call
    tool_message # tool excecution result
]
tool_message
response = model.invoke(messages)
response.content

'<think>\nOkay, the user asked for the weather in Bangalore. I called the get_weather function with location set to Bangalore. The response came back as "Sunny, 33 degree F". Now I need to present this information in a friendly and clear way.\n\nFirst, I should mention the current weather condition, which is sunny. Then the temperature, 33 degrees Fahrenheit. Maybe add a note about what that feels like, since 33°F is a bit cool. Should I mention if it\'s a good temperature for outdoor activities? Also, maybe include a suggestion like bringing a light jacket if they\'re sensitive to the cold. Keep it concise but helpful. Let me check if the units are correct. The user might be from India, so they might be more familiar with Celsius. Wait, the function returned Fahrenheit. Should I convert it? Hmm, but the user didn\'t specify, maybe they\'re using a system that defaults to Fahrenheit. Alternatively, some people in India use Celsius. Maybe I should provide both? Wait, the response says 3